In [3]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

d:\Ai\miniconda3\envs\first\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train = pd.read_csv("../data/train_data.csv")
test = pd.read_csv("../data/test_data.csv")

C:\Users\IT CITY\AppData\Local\Temp\ipykernel_16804\3270014727.py:1: DtypeWarning: Columns (0: vote) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/train_data.csv")


In [ ]:
name = "microsoft/deberta-v3-base"
num_samples = 120000
max_length = 192

train = train.dropna(subset=["overall"])
if len(train) > num_samples:
    train = train.sample(num_samples, random_state=42)

train["label"] = train["overall"].astype(int) - 1
train["text"] = train["summary"].fillna("") + ". " + train["reviewText"].fillna("")
test["text"] = test["summary"].fillna("") + ". " + test["reviewText"].fillna("")

tokenizer = AutoTokenizer.from_pretrained(name)

In [6]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [7]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)


ds = Dataset.from_pandas(train[["text", "label"]], preserve_index=False)
ds = ds.train_test_split(test_size=0.05, seed=42)
ds = ds.map(tokenize, batched=True, remove_columns=["text"])

test_ds = Dataset.from_pandas(test[["text"]], preserve_index=False)
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=5)


def compute_metrics(p):
    pred = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": (pred == p.label_ids).mean(),
        "micro_f1": f1_score(p.label_ids, pred, average="micro"),
    }

Map: 100%|██████████| 20000/20000 [00:01<00:00, 14279.67 examples/s]
d:\Ai\miniconda3\envs\first\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 36469.01it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions

In [ ]:

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/out",
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    output_dir="out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_steps=0.06,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=200,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

val_out = trainer.predict(ds["test"])
val_pred = np.argmax(val_out.predictions, axis=1)
print("micro f1:", f1_score(ds["test"]["label"], val_pred, average="micro"))
print(classification_report(ds["test"]["label"], val_pred))

test_out = trainer.predict(test_ds)
test_pred = np.argmax(test_out.predictions, axis=1) + 1

submission = pd.DataFrame({"predicted": test_pred})
submission.to_csv("q2_submission.csv", index=False)
print(submission.head())
print(len(submission), len(test))

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
d:\Ai\miniconda3\envs\first\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
